# JSE Top 40: three-cross-section study

Clusters the FTSE/JSE Top 40 on financial characteristics at **30 June 2025, 31 December 2025 and 30 June 2026**, and asks whether the peer groups, and the changes in them, are real enough to interpret.

There are 41 instruments per date (40 company groups; Investec has two lines), 123 rows and 43 distinct tickers. Each date is clustered independently. The cross-sections overlap heavily, so they are not independent samples, and nothing here predicts returns.

**To run:** select the project `.venv` kernel, then **Run All** (about two minutes). Every run writes a new folder under `experiments/runs/`; nothing is overwritten. All logic lives in the `top40` package, so this notebook and `python -m top40` produce the same results.
Saved figures and tables are retained in [the executed review](../reports/three_cross_section_review.ipynb).


## 1. Choose the feature set and load the data

`FEATURE_SET` is the one switch that matters:

| Name | Status | What it is |
| --- | --- | --- |
| `original10` | archived reference | The ten inputs behind every earlier result. Includes raw share count and EPS, which change mechanically in a share subdivision. |
| `logsize10` | diagnostic | The same ten with log size variables. |
| `invariant8` | candidate | Nothing changes under a subdivision: return, beta, volatility, dividend yield, earnings yield, log market cap, log revenue/cap, log turnover. |
| `market5` | diagnostic | The five scale-free measures only. |

Everything except `original10` was defined after the earlier results had been seen. Treat those sets as pilot-informed, not pre-registered.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "top40").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from IPython.display import display
from top40.study import StudySession

FEATURE_SET = "invariant8"   # or "original10", "logsize10", "market5"
SEED = 42
STABILITY_RUNS = 5

study = StudySession(feature_set=FEATURE_SET, seed=SEED, stability_runs=STABILITY_RUNS)
data = study.load_data()
DATES = study.config["dates"]
display(data[["decision_date", "ticker", "company", "sector_label", *study.feature_set.columns]].head())

## 2. Exploratory data analysis

Look at the population, the imputed cases and the feature distributions **before** fitting anything. Scaling is fitted on June 2025 only, so the shaded band in the distribution plot is the June 2025 range; points outside it at later dates stretch the geometry.

In [ ]:
eda = study.explore()
display(eda["population"])
display(eda["missingness"].query("raw_missing > 0"))
display(eda["descriptions"][["decision_date", "feature", "min", "50%", "max", "skew"]].round(3))
study.show("eda/population_and_sectors.png")
study.show("eda/feature_distributions.png")

In [ ]:
study.show("eda/membership_presence.png")
display(eda["correlated_pairs"])
for date in DATES:
    study.show(f"eda/correlation_{date}.png")

## 3. Feature validity

Three checks on whether the feature definitions are creating structure.

**Share basis.** Any share count that moves more than 5% between dates, raw and after restating to a common basis using `data/reference/corporate_actions.csv`. Naspers subdivided five-for-one on 6 October 2025: its raw count rises 4.57x while the common-basis count falls to 0.91x.

**Dominance.** The largest share of total scaled variance carried by one feature. Far above the even share means one feature is driving the distances at that date.

**Ablation.** The same recipe on every feature set: the t-SNE baseline, and Ward clustering directly on the scaled features (deterministic, no embedding). Compare cluster sizes, seed agreement, agreement between feature sets, and how many instruments shift. Silhouette and Davies-Bouldin are computed in each set's own space, so do not compare them across feature sets.

In [ ]:
audit = study.audit_features()
display(audit["share_basis"].round(3))
study.show("feature_audit/feature_dominance.png")
display(audit["dominance"].round(3))

In [ ]:
quality_columns = ["feature_set", "recipe", "decision_date", "smallest_cluster", "largest_cluster",
                   "singleton_clusters", "silhouette_features", "mean_seed_ari"]
display(audit["quality"][quality_columns].round(3))
display(audit["cross_set_agreement"].pivot_table(index=["recipe", "feature_set_a", "feature_set_b"],
                                                  columns="decision_date", values="ari").round(3))
display(audit["shift_counts"])
# Which instruments are called stable or shifted depends on the feature set:
status = audit["instrument_status"].set_index("ticker")
display(status[status.nunique(axis=1) > 1])

## 4. Models

The comparison table covers PCA, t-SNE and UMAP with hierarchical, K-Means and DBSCAN clustering at the representative seed. Two **primary models** get full outputs over every seed:

- `baseline`: t-SNE (perplexity 5) then average-linkage Manhattan hierarchy, k=8. This is the recovered recipe of the original paper and is the `tsne_hierarchical` row.
- `umap_sensitivity`: UMAP (3 neighbours, min_dist 0.01) with the same hierarchy. A fixed sensitivity, not a promoted model.

Read silhouette (higher is better) and Davies-Bouldin (lower is better) together, and read the feature-space scores before the embedding scores: t-SNE and UMAP pull points into tight groups, so embedding scores look good even without structure.

In [ ]:
comparison = study.fit_models()
display(comparison[["decision_date", "model", "n_clusters", "coverage", "smallest_cluster", "largest_cluster",
                    "silhouette_features", "davies_bouldin_features",
                    "silhouette_embedding", "davies_bouldin_embedding"]].round(3))
for date in DATES:
    study.show(f"model_quality_{date}.png")

### Maps, profiles and dendrograms

Change `VIEW_DATE` to look at another date. Each map is an independent fit: axes and cluster numbers do not carry over between dates. The profile heatmap (cluster median minus same-date median) is the most direct description of what a cluster is.

In [ ]:
VIEW_DATE = DATES[0]
for name in study.model_folders:
    print(name, VIEW_DATE)
    study.show(f"models/{name}/{VIEW_DATE}/cluster_map.png")
    study.show(f"models/{name}/{VIEW_DATE}/feature_profile.png")
    study.show(f"models/{name}/{VIEW_DATE}/dendrogram.png")

### Seed and cluster-count sensitivity

Mean pairwise ARI between seeds is the only noise reference this notebook computes. It measures how much a partition moves when nothing but the random seed changes. k=8 is the reference specification (the original paper matched eight sector labels), not evidence of eight natural groups: check where the curves actually peak.

In [ ]:
display(study.seed_agreement.groupby(["study_model", "decision_date"])
        .agg(pairs=("ari", "size"), mean_ari=("ari", "mean"), min_ari=("ari", "min")).round(3))
for name in study.model_folders:
    study.show(f"models/{name}/cluster_count_sensitivity.png")

## 5. Explain cluster membership

A gradient-boosted classifier learns each partition from the features, and SHAP explains that classifier. It describes the surrogate, not the clustering and not a cause. Trust it only as far as the **out-of-fold** fidelity allows; in-sample fidelity is always near 1. The table at the end compares SHAP rankings with the plain median deviations from section 4.

In [ ]:
fidelity = study.explain_clusters()
display(fidelity.round(3))
for name in study.model_folders:
    print(name, VIEW_DATE)
    study.show(f"models/{name}/{VIEW_DATE}/shap_global.png")
    study.show(f"models/{name}/{VIEW_DATE}/shap_distribution.png")
    display(study.table(f"models/{name}/{VIEW_DATE}/profile_vs_shap_ranks.csv").round(3))

## 6. Cluster membership across dates

Cluster numbers are aligned date to date by a Hungarian match on Jaccard overlap among instruments present at both dates. Index entries and exits are separate events and never count as a shift.

`changed` counts continuing instruments whose aligned cluster differs. **A changed label is not yet a changed firm**: compare the date-to-date ARI here with the seed-to-seed ARI above. If they are similar, the movement is within fitting noise.

In [ ]:
temporal = study.compare_dates()
display(temporal.round(3))
display(study.longitudinal)

### Coloured cluster flows

One strand per instrument observed at all three dates. Colour is the aligned cluster; a colour change is a shift, and shifted instruments are bold. Vertical position is layout only.

In [ ]:
SHOW_HISTORY_TABLES = False   # set True for the underlying aligned cluster ids
for name in study.model_folders:
    folder = f"longitudinal/{name}"
    study.show(f"{folder}/cluster_flows.png")
    study.show(f"{folder}/jaccard.png")
    study.show(f"{folder}/movement_counts.png")
    events = study.table(f"{folder}/membership_events.csv")
    display(events.query("membership_event != 'continuing'"))
    if SHOW_HISTORY_TABLES:
        display(study.table(f"{folder}/cluster_histories.csv").query("stability_status == 'shifted'"))

### What defines the groups, and what changed?

Profiles show cluster feature deviations alongside within-date surrogate SHAP. Move tables distinguish a firm leaving its peers from a group being redrawn; these are descriptive labels, not causal findings. Read seed support and out-of-fold fidelity alongside each explanation.

The default shows one move per adjacent date pair for each model. Set `MOVE_TICKER` to inspect a particular instrument. All move rows and figures are saved and exported.


In [ ]:
MOVE_TICKER = None   # for example "NPN"; None shows one move per date pair
for name in study.model_folders:
    folder = f"longitudinal/{name}/moves"
    print(name, "cluster definitions")
    profiles = study.table(f"{folder}/cluster_summary.csv")
    display(profiles)
    study.show(f"{folder}/cluster_profiles_{VIEW_DATE}.png")
    moves = study.table(f"{folder}/moves.csv")
    print(name, "all observed aligned-label moves")
    display(moves)
    selected = (moves[moves.ticker == MOVE_TICKER] if MOVE_TICKER else
                moves.groupby(["before", "after"], sort=False).head(1))
    for move in selected.itertuples():
        study.show(f"{folder}/move_{move.before}_{move.after}_{move.ticker}.png")


### Distance screen and sector context

The screen is the squared Mahalanobis distance of each instrument from the same-date mean, in the scaled feature space, against the exact Beta reference for an individual same-sample distance under normality. Equity cross-sections are not normal, so more than 5% will exceed it: read it as a ranking of unusual profiles, not a test. It does not depend on the clustering model.

In [ ]:
screens = study.screens
display(screens.groupby("decision_date").agg(flagged=("flag", "sum"), reference=("reference", "first")).round(2))
display(screens[screens.flag == 1].sort_values(["decision_date", "squared_distance"], ascending=[True, False]).round(2))
display(study.table("longitudinal/baseline/sector_context.csv").query("sector_status != 'compared'"))

## 7. Finish the run

The run is marked completed only after every expected output exists. `summary.md` in the run folder collects the main tables.

In [ ]:
from IPython.display import Markdown
run_folder = study.finish()
display(Markdown((run_folder / "summary.md").read_text(encoding="utf-8")))

## What this does and does not establish

- The clusters are descriptive groupings of reconstructed, partly unverified inputs. Share counts, market capitalisation and sector labels are not verified point-in-time.
- No external validation is done here. The planned test is whether clusters formed at one date explain return co-movement over the following six months better than sector labels; that needs daily prices, which are not in this repository.
- The noise reference is seed agreement only. A perturbation-based noise floor (resampling instruments) is still to be added.